# 3-5. Self-RAG / CRAG on LangGraph

⏱ **소요시간**: 30분

## 학습 목표
- LangGraph `StateGraph`와 `TypedDict` 상태로 **자가교정 RAG**를 구현할 수 있다.
- Self-RAG의 문서 관련성 평가 → (관련 있으면) 생성 / (없으면) 쿼리 재작성 후 재검색 루프를 설계할 수 있다.
- Self-RAG와 CRAG (Corrective RAG)의 차이를 설명할 수 있다.

## 핵심 키워드
LangGraph · StateGraph · TypedDict · Self-RAG · CRAG · 자가교정


In [ ]:
# 공통 세팅: common 헬퍼를 사용하기 위해 repo 루트를 sys.path에 추가
import sys, os
sys.path.insert(0, '../..')

from common import get_llm, get_embeddings, provider_badge
from common.ko_tokenizer import tokenize_ko
from common.loaders import load_any

print(provider_badge())


## 1. 개념 비교: Self-RAG vs CRAG

| | Self-RAG | CRAG (Corrective RAG) |
|---|---------|-----------------------|
| 평가 주체 | LLM이 검색 결과와 자체 생성물을 reflection token으로 평가 | 별도의 경량 평가기(retrieval evaluator) |
| 낮은 품질 시 | 재검색 또는 검색 생략(자체 지식 사용) | 웹 검색/쿼리 재작성으로 보완 |
| 강점 | 세밀한 토큰 단위 제어 | 구현이 단순, 실패 복구 명확 |
| 약점 | fine-tuning된 모델 필요 | 평가기 성능에 의존 |

이 노트북에서는 **LangGraph로 미니 Self-RAG**를 만든다. fine-tuned 모델 대신 일반 LLM에 few-shot 프롬프트로 grader 역할을 맡긴다.

## 2. 상태 기계 그래프 설계

```
        ┌──────────┐
        │ retrieve │
        └────┬─────┘
             ▼
      ┌──────────────┐      관련 없음     ┌──────────────┐
      │ grade_docs   │──────────────────▶│ rewrite_query│
      └──────┬───────┘                   └──────┬───────┘
             │ 관련 있음                         │
             ▼                                  │
        ┌──────────┐                            │
        │ generate │◀───────────────────────────┘
        └──────────┘             (재시도)
```


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

# 공통 인덱스 준비
docs = load_any('../../data/corpus_ko.txt')
chunks = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40).split_documents(docs)
vectordb = FAISS.from_documents(chunks, get_embeddings())
retriever = vectordb.as_retriever(search_kwargs={'k': 3})
llm = get_llm(temperature=0.0)


## 3. 상태 정의와 노드 구현

In [ ]:
from typing import TypedDict, Annotated
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

class RagState(TypedDict):
    question: str             # 현재(재작성 포함) 질문
    original_question: str    # 사용자가 처음 준 질문
    documents: list[Document]
    generation: str
    retries: int              # 재시도 횟수 상한용
    log: list[str]            # 디버그 로그


In [ ]:
# --- 노드 1: retrieve ---
def retrieve(state: RagState) -> RagState:
    q = state['question']
    docs = retriever.invoke(q)
    state['documents'] = docs
    state['log'] = state.get('log', []) + [f'[retrieve] q="{q}" -> {len(docs)}건']
    return state


In [ ]:
# --- 노드 2: grade_documents ---
grade_prompt = ChatPromptTemplate.from_template(
    '다음 문서가 질문에 답하는 데 **관련 있는지** 판단하라.\n'
    '관련 있으면 yes, 없으면 no 한 단어만 출력.\n\n질문: {q}\n문서: {d}'
)
grade_chain = grade_prompt | llm | StrOutputParser()

def grade_documents(state: RagState) -> RagState:
    q = state['question']
    good = []
    for d in state['documents']:
        verdict = grade_chain.invoke({'q': q, 'd': d.page_content}).strip().lower()
        if verdict.startswith('yes'):
            good.append(d)
    state['documents'] = good
    state['log'].append(f'[grade] 관련 문서 {len(good)}건 유지')
    return state


In [ ]:
# --- 노드 3: rewrite_query ---
rewrite_prompt = ChatPromptTemplate.from_template(
    '다음 질문을 더 명확하고 검색이 잘 되도록 다시 써라. 한 줄만 출력:\n{q}'
)
rewrite_chain = rewrite_prompt | llm | StrOutputParser()

def rewrite_query(state: RagState) -> RagState:
    new_q = rewrite_chain.invoke({'q': state['question']}).strip()
    state['question'] = new_q
    state['retries'] = state.get('retries', 0) + 1
    state['log'].append(f'[rewrite] 새 질문: "{new_q}" (retry={state["retries"]})')
    return state


In [ ]:
# --- 노드 4: generate ---
gen_prompt = ChatPromptTemplate.from_template(
    '아래 컨텍스트만 사용해 한국어로 답하라. 근거가 없으면 "근거 없음"이라고 답하라.\n\n'
    '컨텍스트:\n{ctx}\n\n질문: {q}\n답변:'
)
gen_chain = gen_prompt | llm | StrOutputParser()

def generate(state: RagState) -> RagState:
    ctx = '\n---\n'.join(d.page_content for d in state['documents'])
    ans = gen_chain.invoke({'ctx': ctx, 'q': state['original_question']})
    state['generation'] = ans
    state['log'].append(f'[generate] {len(ans)}자 답변 생성')
    return state


In [ ]:
# --- 분기 함수: grade 후 어디로 갈지 결정 ---
def decide_after_grade(state: RagState) -> str:
    if len(state['documents']) == 0 and state.get('retries', 0) < 2:
        return 'rewrite'
    return 'generate'


## 4. StateGraph 조립 및 컴파일

In [ ]:
from langgraph.graph import StateGraph, END

g = StateGraph(RagState)
g.add_node('retrieve', retrieve)
g.add_node('grade', grade_documents)
g.add_node('rewrite', rewrite_query)
g.add_node('generate', generate)

g.set_entry_point('retrieve')
g.add_edge('retrieve', 'grade')
g.add_conditional_edges('grade', decide_after_grade, {
    'rewrite': 'rewrite',
    'generate': 'generate',
})
g.add_edge('rewrite', 'retrieve')   # 다시 검색
g.add_edge('generate', END)

app = g.compile()
print('Self-RAG 그래프 컴파일 완료')


## 5. End-to-End 실행 & 로그 확인

In [ ]:
question = '망분리는 왜 금융권에서 의무화되어 있는가?'
init_state: RagState = {
    'question': question,
    'original_question': question,
    'documents': [],
    'generation': '',
    'retries': 0,
    'log': [],
}
final = app.invoke(init_state)

print('=== 실행 로그 ===')
for line in final['log']:
    print(line)
print('\n=== 최종 답변 ===')
print(final['generation'])


## 6. CRAG로 확장하려면?

이 그래프에 아래 노드를 하나 추가하면 CRAG 스타일이 된다.

- `web_search` 노드: 검색 품질이 낮을 때 외부 검색(airgap 환경에서는 사내 검색 엔진 / 다른 벡터스토어)으로 보강.
- 경량 `retrieval_evaluator` 노드: T5 기반 소형 분류 모델로 문서 관련성 점수 산출 (LLM 호출 비용 절감).

> 폐쇄망에서는 "외부 웹" 대신 **다른 사내 지식베이스**(예: 사내 Wiki Qdrant, 사내 JIRA)로 fallback하는 패턴이 현실적이다.


## 과제
1. `retries` 상한을 1/2/3으로 바꾸며 답변 품질과 지연을 기록하라.
2. `grade_documents`의 프롬프트를 "no" 쪽으로 보수적/공격적으로 튜닝해 행동 변화를 관찰하라.
3. `generate` 후에 **답변-컨텍스트 일치성**을 검사하는 노드(`check_hallucination`)를 추가해 완전한 Self-RAG 루프를 완성하라.

## 더 읽어보기
- LangChain Korea: <https://github.com/teddylee777/langchain-kr> → `17-LangGraph` 전체 챕터
- Self-RAG 원논문: *Self-RAG: Learning to Retrieve, Generate, and Critique* (Asai et al., 2023)
- CRAG 원논문: *Corrective Retrieval Augmented Generation* (Yan et al., 2024)
- LangGraph 공식 예제: <https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph_self_rag/>
